Cell 1: Setup

In [1]:
import re 
import sys

import pandas as pd

pd.set_option("display.max_colwidth", 200)

df_raw = pd.read_csv("../data/raw/fctr.csv", sep="\t")
print("Raw data shape: ", df_raw.shape)

Raw data shape:  (3238, 7)


In [2]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 3238 entries, 0 to 3237
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   claim_id  3238 non-null   int64
 1   claim     3238 non-null   str  
 2   evidence  3238 non-null   str  
 3   summary   3238 non-null   str  
 4   label     3238 non-null   str  
 5   url       3238 non-null   str  
 6   date      3238 non-null   str  
dtypes: int64(1), str(6)
memory usage: 12.6 MB


Cell 2: A helper to inspect regex matches

In [3]:
def show_matches(texts, pattern, n=15, width=30, seed=42):
    """Print a random sample of regex matches with surrounding context."""
    snippets = []
    for text in texts:
        for match in re.finditer(pattern, text):
            start = max(match.start() - width, 0)
            snippets.append(text[start:match.end() + width])

    print(f"Total Matches: {len(snippets)}\n")
    sample = pd.Series(snippets).sample(min(n, len(snippets)), random_state=seed)
    for snippet in sample:
        print("...", snippet, "...")

Cell 3: Glued sentence boundaries

In [4]:
show_matches(df_raw["evidence"], r"[.!?][A-ZÇĞİÖŞÜ]")

Total Matches: 13710

... i kurumlar karar veriyor” dedi.Demirtaş’ın kitaplarına, Idefix ...
...  BU iS OLMAZ DiYE DUSUNMEYiN...LUTFEN, VATANINI SEVEN HERKESE  ...
...  biri olmakla itham etmekteydi.Gülbeddin Hikmetyar ise 1977 yı ...
... eflerinde belirtildiği söyledi.Cinsiyet Eşitliği Kurumu’nun ya ...
... yretmek üzere ikiye ayrılırlar.Yani, sol elin dördüncü parmağı ...
...  da geçmesi de şaşırtıcı değil.Şahin’in açıklamasında aşı prot ...
... iada da bu video kullanılmıştı.Görüntüler The Simpsons’ın 6. s ...
... ariyle bu program durdurulacak.Program tarafından sunulan araş ...
... aların devam ettiği belirtildi.Kuleli Askeri Lisesi satılmadıK ...
... otokol bulunmadığını belirtmiş.Bilgilendirme seminerini bir şi ...
... ter’dan bu hesabı takip ediyor.Aynı isimli Instagram hesabının ...
... olduğunu belirterek başlayalım.İddialarda yer alan kısa video  ...
... nıtlanmış olmadığını görüyoruz.Örneğin 2016 tarihli bu çalışma ...
... iki binin üzerinde beğeni aldı.Ancak iddia doğru de

Cell 4: Glued headings

In [5]:
show_matches(df_raw["evidence"], r"[a-zçğıöşü]{3}[A-ZÇĞİÖŞÜ][a-zçğıöşü]")

Total Matches: 5117

... larda yolunu kaybeden gemi misaliRotasız ve pusulasız kalmışken y ...
... göre MRna aşılarının yan etkileriAmerikan Gıda ve İlaç Dairesi’ni ...
... NTech aşısının içeriğini açıkladıAmerikan Gıda ve İlaç Birimi’nin ...
... eyit’e gönderdiği kira sözleşmesiİddia daha önce Azerbaycan dilin ...
... ına aldırış etmedi.” – TyC SportsHaberin içeriğini detaylı bir şe ...
... montajlanmış.Görüntü montajlanmışBakan Blanquer’ın 4 Mayıs 2020’d ...
... ylüyor.Doğrulama platformu PolitiFact ve Snopes da Dr. Stella Imm ...
... ia edildi.FDA verileri gizlemiyorSosyal medyada paylaşılan iddial ...
... macunu tüpüc)Deodorant d)El kremiSosyal medyada sıkça paylaşılan  ...
... di.Fotoğraf Şanlıurfa’da çekilmişSosyal medyada iddia ile paylaşı ...
... ramalarda Greta Thunberg’in KyodoNews’e verdiği röportaja ulaşılı ...
... an yönetimi altında kadın haklarıTaliban, 1990’lı yılların sonlar ...
... s Downtown AntalyaTürkiye-AlmanyaEkran görüntüsü 18.05.2021 tarih ...
... .  Youtube, M

Cell 5: Find frequent CamelCase words

In [6]:
camel_words = (
    df_raw["evidence"]
    .str.findall(r"\b\w*[a-zçğıöşü][A-ZÇĞİÖŞÜ]\w*\b")
    .explode()
    .value_counts()
)
camel_words.head(40)

evidence
YouTube            406
BioNTech           298
CoV                294
mRNA               283
WhatsApp           224
TikTok              90
AstraZeneca         85
HaberTürk           64
pH                  57
PolitiFact          51
CoronaVac           40
kWh                 26
BirGün              25
McCain              24
McDonald            21
PubMed              19
BionTech            18
SpaceX              17
iPhone              16
LinkedIn            15
FactCheck           13
CryptoRobo          13
GlaxoSmithKline     11
CerModern           11
yokTeyit            10
eBay                 9
iData                9
YouTuber             9
TinEye               9
FlightRadar24        9
FullFact             9
MacMillan            9
yokİddia             8
bizHer               8
ulaştıTeyit          8
CanSino              8
iddiasıİlginizi      8
NeoCov               7
VigiAccess           7
değilTeyit           7
Name: count, dtype: int64

In [8]:
heading_pattern = r"[a-zçğıöşü]{3}[A-ZÇĞİÖŞÜ][a-zçğıöşü]"

# CamelCase words that our heading rule WOULD split, appearing at least 3 times
at_risk = camel_words[camel_words.index.str.contains(heading_pattern, regex=True)]
review = at_risk[at_risk >= 3]
for word, count in review.items():
    print(f"{count:4}  {word}")
    

 224  WhatsApp
  85  AstraZeneca
  64  HaberTürk
  51  PolitiFact
  40  CoronaVac
  18  BionTech
  15  LinkedIn
  13  FactCheck
  13  CryptoRobo
  11  GlaxoSmithKline
  10  yokTeyit
   9  FlightRadar24
   9  FullFact
   8  yokİddia
   8  bizHer
   8  ulaştıTeyit
   8  iddiasıİlginizi
   7  VigiAccess
   7  değilTeyit
   7  BlackRock
   7  KaynakSonuç
   6  OceanGate
   6  değilSosyal
   6  yokSosyal
   6  yokTürkiye
   6  AzizGwenSancar
   6  KaynakÖte
   6  FransaEkran
   5  değilİddiada
   5  BioCubaFarma
   5  iddiasıSonuç
   5  ÇapaMag
   5  NewsPunch
   5  AlmanyaEkran
   5  AvustralyaEkran
   5  RusyaEkran
   4  GenclikHudPar
   4  söylediTeyit
   4  bizBaşarının
   4  yoluyuzYine
   4  değilArama
   4  koyamadığımdınSenin
   4  yokPeki
   4  montajlanmışAncak
   4  BuzzFeed
   4  değilAncak
   4  MarineTraffic
   4  BelkoAir
   4  SheikhResit
   4  EctoLife
   4  MythDetector
   4  foundDosyayı
   3  yokKonuyla
   3  montajlanmışİddia
   3  SpecTor
   3  doğruladıTeyit
   3  değ

Cell 6: Import the cleaning module

In [9]:
%load_ext autoreload
%autoreload 2

sys.path.append("..")
from src.data_cleaning import clean_text, clean_fctr

Cell 7: Before and after on real examples

In [13]:
examples = df_raw[df_raw["claim_id"].isin([2, 6680, 431])]

for _, row in examples.iterrows():
    print(f"claim_id {row['claim_id']}")
    print("BEFORE:", row["evidence"][:250], "...", row["evidence"][-120:])
    cleaned = clean_text(row["evidence"])
    print("AFTER:", cleaned[:250], "...", cleaned[-120:])
    print("-" * 100)

claim_id 2
BEFORE: Sosyal medyada sık sık yapılan paylaşımlarla Amazon’un sahiplerine ulaşamayan paketleri hediye ettiği veya cüzi miktarlarda sattığı iddia edildi.İddialara göre Amazon, teslim edilemeyen çeşitli ürünler içeren paketlerini kullanıcılara 1,95 euro karşı ... li bir siteden kullanıcılar satıcıların yorumlarını okuyabiliyor ve GovDeals gibi daha köklü satıcılara ulaşabiliyorlar.
AFTER: Sosyal medyada sık sık yapılan paylaşımlarla Amazon’un sahiplerine ulaşamayan paketleri hediye ettiği veya cüzi miktarlarda sattığı iddia edildi. İddialara göre Amazon, teslim edilemeyen çeşitli ürünler içeren paketlerini kullanıcılara 1,95 euro karş ... li bir siteden kullanıcılar satıcıların yorumlarını okuyabiliyor ve GovDeals gibi daha köklü satıcılara ulaşabiliyorlar.
----------------------------------------------------------------------------------------------------
claim_id 431
BEFORE: Gazeteci Cansu Çamlıbel, 14 Şubat 2023’te İTÜ Jeoloji Mühendisliği Bölümü'nden Prof. Dr. Cenk Yaltır

Cell 8: Clean the whole dataset

In [14]:
df = clean_fctr(df_raw)
print("Clean shape:", df.shape)
df.head()

Clean shape: (3238, 15)


,doc_id,claim_id,source,source_name,date,claim,evidence,summary,label_original,label,verdict_group,binary_label,is_headline_claim,possibly_truncated,url
0,fctr-2,2,teyit.org,Teyit,2023-07-11,Amazon'un Türkiye'de teslim edilmemiş ürünleri uygun fiyata satışa çıkardığı iddiası,"Sosyal medyada sık sık yapılan paylaşımlarla Amazon’un sahiplerine ulaşamayan paketleri hediye ettiği veya cüzi miktarlarda sattığı iddia edildi. İddialara göre Amazon, teslim edilemeyen çeşitli ü...","Amazon Türkiye’de teslim edilemeyen paketlerin yeniden satışa çıkarılması gibi bir uygulama yok. ABD’de yürütülen bu uygulamada satışlar, Amazon’dan bağımsız ve yetkili kanallar aracılığıyla yapıl...",yanlış,yanlış,FALSE,FALSE,False,False,https://teyit.org/analiz/amazonun-turkiyede-teslim-edilmemis-urunleri-uygun-fiyata-satisa-cikardigi-iddiasi
1,fctr-5,5,teyit.org,Teyit,2023-07-11,Mickey Mouse’un penisiyle peynir deldiği film Disney’e mi ait?,Walt Disney'in en çok tanınan karakterlerinden olan Mickey Mouse'a ait bir bölümde müstehcen görüntüler olduğu iddia edildi. Videoda Mickey'nin İsviçre peynirine penisiyle delik açtığı görülüyor. ...,İddia görüntü dijital içerik üreten kullanıcı tarafından oluşturulmuş. Orijinal bölümde benzeri bir içerik yer almıyor.,yanlış,yanlış,FALSE,FALSE,False,False,https://teyit.org/analiz/mickey-mouseun-penisiyle-peynir-deldigi-film-disneye-mi-ait
2,fctr-6,6,teyit.org,Teyit,2023-07-11,Kurtlar Vadisi'nde Milli Muharip Uçak'ın öngörüldüğü iddiası,"Sosyal medyada bazı kullanıcılar Kurtlar Vadisi dizisinin 2009’da yayınlanan bir bölümünde, Türk Havacılık ve Uzay Sanayii AŞ (TUSAŞ) tarafından Milli Muharip Uçak projesi altında üretilen Kaan is...","Dizide kullanılan görsel Joint Strike Fighter projesi kapsamında üretilen F-35C’ye ait. Kaan isimli savaş uçağının fotoğrafı, dizide kullanılan F-35C’ye benzeyecek şekilde montajlanmış.",yanlış,yanlış,FALSE,FALSE,False,False,https://teyit.org/analiz/kurtlar-vadisinde-milli-muharip-ucakin-gorundugu-iddiasi
3,fctr-7,7,teyit.org,Teyit,2023-07-10,Basketbol topu şeklindeki binanın gerçek olduğu iddiası,"Sosyal medyada yapılan bir paylaşımda, görüntülerdeki binanın bilgisayar teknolojisi ile tasarlanmamış gerçek bir bina olduğu iddia edildi. Bazı sosyal medya kullanıcıları ise görüntülerin CGI tek...","Görüntülerdeki bina “Madison Square Garden Sphere” olarak isimlendirilen bir eğlence merkezi. Binanın 2,3 milyar dolara yapıldığı tahmin ediliyor. Binanın açılışı 2023’ün sonlarında yapılacak. Dış...",doğru,doğru,TRUE,TRUE,False,False,https://teyit.org/analiz/basketbol-topu-seklindeki-binanin-gercek-oldugu-iddiasi
4,fctr-13,13,teyit.org,Teyit,2023-07-10,Threads'in logosunun Homer Simpson'ın kulaklarından esinlenildiği iddiası,"5 Temmuz 2023’te Instagram, yeni bir mikroblog uygulaması olan Threads’i tanıttı. Threads logo dizaynıyla ilgili henüz resmi bir açıklama yapılmasa da internet kullanıcıları logonun neye benzediği...","Homer Simpson’ın orijinal çizimlerindeki kulakları, Threads logosuna benzemiyor. Homer’in kulaklarını Threads logosuna benzeterek paylaşılan fotoğraflar manipüle edilmiş.",yanlış,yanlış,FALSE,FALSE,False,False,https://teyit.org/analiz/threadsin-logosunun-homer-simpsonin-kulaklarindan-esinlenildigi-iddiasi


Cell 9: Measure the effect

In [16]:
glued = r"[.!?][A-ZÇĞİÖŞÜ]"
heading = r"[a-zçğıöşü]{3}[A-ZÇĞİÖŞÜ][a-zçğıöşü]"

for name, pattern in [("glued sentences", glued), ("glued headings", heading)]:
    before = df_raw["evidence"].str.count(pattern).sum()
    after = df["evidence"].str.count(pattern).sum()
    print(f"{name:16}: {before:6} -> {after}")

before_links = df_raw["evidence"].str.contains("İlginizi çekebilir:", regex=False).sum()
after_links = df["evidence"].str.contains("İlginizi çekebilir:", regex=False).sum()
print(f"related links   : {before_links:6} -> {after_links}")

print("possibly truncated:", df["possibly_truncated"].sum())

glued sentences :  13710 -> 0
glued headings  :   5117 -> 11
related links   :    168 -> 133
possibly truncated: 87


Cell 10: Check the label mapping

In [17]:
print(pd.crosstab(df["label"], df["verdict_group"]))
print()
print(df["binary_label"].value_counts(dropna=False))

assert df["verdict_group"].notna().all(), "Some labels were not mapped!"

verdict_group       FALSE  MIXED  TRUE  UNPROVEN
label                                           
doğru                   0      0   203         0
karma                   0    109     0         0
kısmen yanlış           0     72     0         0
sonuçlandırılamadı      0      0     0        37
yanlış               2780      0     0         0
yarı doğru              0     17     0         0
çoğunlukla doğru        0      0     6         0
çoğunlukla yanlış      14      0     0         0

binary_label
FALSE    2780
NaN       255
TRUE      203
Name: count, dtype: int64


Cell 11: Inspect headline-style claims

In [18]:
print(df["is_headline_claim"].value_counts())
print()
print(pd.crosstab(df["source_name"], df["is_headline_claim"]))

df[df["is_headline_claim"]][["claim", "label", "source_name"]].sample(15, random_state=1)

is_headline_claim
False    2599
True      639
Name: count, dtype: int64

is_headline_claim  False  True 
source_name                    
Doğrula              343    182
Doğruluk Payı        533    209
Teyit               1723    248


,claim,label,source_name
2382,Bir Twitter hesabı tarafından 4 Şubat 2022 tarihinde yapılan paylaşımda 3 Şubat 2022’de Samsun’daki Onur Anıtına (Atatürk Anıtı) halat bağlayıp yıkmaya çalışan kişinin CHP Samsun İl Başkan Yardımc...,yanlış,Doğruluk Payı
2948,Terk edilen Rus askeri araçları kullandığı iddia edilen Ukraynalı kadın,yanlış,Doğrula
3049,Cocostar’da beyazlatılmış havuç yok,yanlış,Doğrula
2688,"Bir Instagram kullanıcısı 7 Eylül 2020 tarihinde paylaşılan gönderide Türkiye’de biraya, sigaraya, akaryakıtlara, arabalara ve maaşlara uygulanan vergilere dair çeşitli iddialarda bulundu. Paylaşı...",kısmen yanlış,Doğruluk Payı
2997,‘İdrar’ Covid-19’a karşı kullanılan bir tedavi yöntemi değil,yanlış,Doğrula
3067,Renkli rozet sözde”Yeni Dünya Düzeni” projesinin üyeliğini göstermiyor,yanlış,Doğrula
2697,"Bir Instagram sayfası tarafından 13 Ağustos 2020 tarihinde yapılan paylaşımda, Cumhurbaşkanı Recep Tayyip Erdoğan’ın, ASELSAN’ı biz kurduk dediği iddia edildi. Gönderi 4 bin 500’e yakın beğeni aldı.",yanlış,Doğruluk Payı
3088,‘İdeal Cumhuriyet Köyü’ projesi ismi bilinmeyen bir mimar tarafından çizildi,yanlış,Doğrula
2478,Bir Twitter hesabı tarafından 13 Ekim 2021 tarihinde yapılan paylaşımda Delta Havayolları'nda çalışan aşılı bir pilotun uçuş sırasında öldüğü iddia edildi. Gönderi 160’dan fazla retweet 450’den fa...,yanlış,Doğruluk Payı
1578,NASA vs Diyanet - 4: Kadının toplumsal rolüne bakış açısından karşılaştırmalar,doğru,Teyit


Cell 12: Save and verify

In [19]:
output_path = "../data/processed/fctr_clean.parquet"
df.to_parquet(output_path, index=False)

# Read it back to make sure types survived the round trip
check = pd.read_parquet(output_path)
print(check.shape)
print(check.dtypes)

(3238, 15)
doc_id                           str
claim_id                       int64
source                           str
source_name                      str
date                  datetime64[us]
claim                            str
evidence                         str
summary                          str
label_original                   str
label                            str
verdict_group                    str
binary_label                     str
is_headline_claim               bool
possibly_truncated              bool
url                              str
dtype: object
